# 多因子模型比较完整教程：从 GRS 检验到模型选择

## 📚 教学目标
1. 理解为什么需要**联合检验**所有 alpha，而非逐个检验
2. 掌握 **GRS 检验 (Gibbons-Ross-Shanken Test)** 的完整推导与手算
3. 使用 GRS 检验比较 **CAPM、Fama-French 三因子、五因子模型**
4. 理解 **平均绝对 alpha ($|\alpha|$)** 作为模型比较的辅助指标
5. 建立"好模型 = 小 GRS + 小 $|\alpha|$"的直觉

**参考书**：《因子投资：方法与实践》（石川）第 2.5 节
**教学策略**：先用 5 个资产的微型例子手算 GRS 的每一步，再扩展到 25 个资产比较三个模型

---

## 1. 为什么要比较多因子模型？

### 🎯 场景设定

假设你已经构建了三个多因子模型来解释股票收益率的截面差异：

| 模型 | 因子 | 因子数 |
|------|------|--------|
| CAPM | MKT | 1 |
| Fama-French 三因子 | MKT, SMB, HML | 3 |
| Fama-French 五因子 | MKT, SMB, HML, RMW, CMA | 5 |

**核心问题**：哪个模型最好？

### 📖 书中要点

> 多因子模型的优劣可通过其对检验资产（test assets）截面收益差异的解释能力来评判。
> 如果一个模型是正确的，那么所有检验资产相对于该模型的 alpha 应该联合为零。

### 💡 直觉

```
好模型 → 能解释所有资产的收益 → 所有 alpha ≈ 0
差模型 → 遗漏了重要因子 → 某些资产有显著的 alpha

模型比较 = 看谁的 alpha 更接近全零
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 单个 alpha 检验 vs 联合检验

### 🎯 为什么不能逐个检验 alpha？

假设我们有 $N$ 个检验资产，对每个资产 $i$ 做时间序列回归：

$$r_{i,t} - r_{f,t} = \alpha_i + \boldsymbol{\beta}_i' \mathbf{f}_t + \varepsilon_{i,t}$$

得到 $N$ 个 alpha 及其 t 统计量。**逐个检验**的问题：

```
单个检验:  H₀: α₁ = 0  →  t₁ = 1.8  →  不显著 ✓
单个检验:  H₀: α₂ = 0  →  t₂ = 1.5  →  不显著 ✓
          ...
单个检验:  H₀: αN = 0  →  tN = 2.1  →  显著 ✗

问题: 25 个检验中出现 1 个"显著"结果可能纯属偶然！
```

### 📐 多重检验问题 (Multiple Testing Problem)

当同时进行 $N$ 个独立假设检验时，至少出现一个错误拒绝的概率为：

$$P(\text{至少一个误判}) = 1 - (1 - \alpha)^N$$

| $N$ (检验数) | $\alpha = 0.05$ | 误判概率 |
|:---:|:---:|:---:|
| 1 | 0.05 | 5% |
| 5 | 0.05 | 22.6% |
| 10 | 0.05 | 40.1% |
| 25 | 0.05 | 72.3% |

25 个资产逐个检验时，即使所有 alpha 都为零，也有 **72.3%** 的概率看到至少一个"显著"结果！

### 💡 解决方案：联合检验

GRS 检验一次性检验所有 alpha 是否**联合为零**：

$$H_0: \alpha_1 = \alpha_2 = \cdots = \alpha_N = 0$$

只输出**一个**检验统计量和**一个** P 值，避免了多重检验问题。

In [ ]:
# ========== 微型示例：5 个检验资产，看逐个检验的问题 ==========
np.random.seed(42)
T_micro = 60   # 60 个月
N_micro = 5    # 5 个检验资产
K_micro = 1    # 1 个因子 (MKT)

# 生成因子收益
MKT = np.random.normal(0.5, 4.0, T_micro)  # MKT ~ N(0.5%, 4%)

# 真实 alpha: 不为零（模型是错的）
true_alphas = np.array([0.8, 0.5, -0.3, 0.6, -0.4])  # 月均 %
true_betas = np.array([0.8, 1.0, 1.2, 0.6, 1.4])

# 生成检验资产收益
epsilon = np.random.normal(0, 3.0, (T_micro, N_micro))  # 残差
R = np.zeros((T_micro, N_micro))
for i in range(N_micro):
    R[:, i] = true_alphas[i] + true_betas[i] * MKT + epsilon[:, i]

print(f"📊 微型示例参数：")
print(f"  检验资产数 N = {N_micro}")
print(f"  时间期数 T = {T_micro}")
print(f"  因子数 K = {K_micro} (仅 MKT)")
print(f"  真实 alpha = {true_alphas}")
print(f"  真实 beta  = {true_betas}")

# ========== 逐个回归并检验 alpha ==========
print(f"\n📊 逐个时间序列回归结果：")
print("─" * 65)
print(f"  {'资产':>4}  {'α̂':>8}  {'t(α)':>8}  {'P值':>10}  {'单独显著?':>10}")
print("─" * 65)

alpha_hat = np.zeros(N_micro)
residuals = np.zeros((T_micro, N_micro))

X = sm.add_constant(MKT)  # 加截距项
for i in range(N_micro):
    model = sm.OLS(R[:, i], X).fit()
    alpha_hat[i] = model.params[0]
    residuals[:, i] = model.resid
    t_alpha = model.tvalues[0]
    p_alpha = model.pvalues[0]
    sig = "✅ 显著" if p_alpha < 0.05 else "❌ 不显著"
    print(f"  资产{i+1}  {alpha_hat[i]:>8.4f}  {t_alpha:>8.3f}  {p_alpha:>10.4f}  {sig:>10}")

n_sig = sum(1 for i in range(N_micro)
            if sm.OLS(R[:, i], X).fit().pvalues[0] < 0.05)
print("─" * 65)
print(f"\n💡 {n_sig}/{N_micro} 个 alpha 单独显著")
print(f"   但这不能告诉我们：这些 alpha 作为整体是否显著不为零")
print(f"   需要 GRS 联合检验！")

---

## 3. GRS 检验 (Gibbons-Ross-Shanken Test)

### 📐 GRS 检验统计量

$$\text{GRS} = \frac{T - N - K}{N} \cdot \frac{\hat{\boldsymbol{\alpha}}' \hat{\boldsymbol{\Sigma}}_\varepsilon^{-1} \hat{\boldsymbol{\alpha}}}{1 + \hat{\boldsymbol{\mu}}_f' \hat{\boldsymbol{\Sigma}}_f^{-1} \hat{\boldsymbol{\mu}}_f} \sim F(N, T - N - K)$$

其中：
- $T$ = 时间期数
- $N$ = 检验资产数
- $K$ = 因子数
- $\hat{\boldsymbol{\alpha}}$ = 回归 alpha 向量 ($N \times 1$)
- $\hat{\boldsymbol{\Sigma}}_\varepsilon$ = 残差协方差矩阵 ($N \times N$)
- $\hat{\boldsymbol{\mu}}_f$ = 因子均值向量 ($K \times 1$)
- $\hat{\boldsymbol{\Sigma}}_f$ = 因子协方差矩阵 ($K \times K$)

### 💡 直觉理解

GRS 统计量的两个核心组成部分：

| 部分 | 公式 | 含义 |
|------|------|------|
| 分子 | $\hat{\boldsymbol{\alpha}}' \hat{\boldsymbol{\Sigma}}_\varepsilon^{-1} \hat{\boldsymbol{\alpha}}$ | alpha 的"加权平方和"——衡量 alpha 整体偏离零的程度 |
| 分母 | $1 + \hat{\boldsymbol{\mu}}_f' \hat{\boldsymbol{\Sigma}}_f^{-1} \hat{\boldsymbol{\mu}}_f$ | $1 + \text{SR}^2_f$——因子组合最大 Sharpe Ratio 的平方 |

### 📖 书中解释

> GRS 检验的本质是判断：在检验资产中是否存在模型未能解释的定价误差。
> 如果模型是正确的，所有 alpha 联合为零，GRS 统计量应接近 1。

In [ ]:
# ========== 手算 GRS：在微型示例上逐步计算 ==========
T = T_micro
N = N_micro
K = K_micro

print("📊 步骤 1: 收集回归结果")
print(f"  alpha 向量 α̂ = {np.round(alpha_hat, 4)}")
print(f"  维度: ({N},)")
print()

# ========== 步骤 2: 计算残差协方差矩阵 Σ_ε ==========
# 使用 ML 估计 (除以 T，而非 T-K-1)，与 GRS 原始论文一致
Sigma_eps = (residuals.T @ residuals) / T

print("📊 步骤 2: 残差协方差矩阵 Σ̂_ε")
print(f"  公式: Σ̂_ε = (1/T) Σ ε̂_t ε̂_t'")
print(f"  维度: ({N}, {N})")
print(f"  对角线元素 (各资产残差方差):")
for i in range(N):
    print(f"    资产{i+1}: σ²_ε = {Sigma_eps[i, i]:.4f}")
print()

# ========== 步骤 3: 计算因子均值和协方差 ==========
factors = MKT.reshape(-1, 1)  # (T, K)
mu_f = factors.mean(axis=0)    # (K,)
Sigma_f = np.cov(factors.T, ddof=0)  # (K, K)，ML 估计
if Sigma_f.ndim == 0:
    Sigma_f = np.array([[Sigma_f]])  # 确保是矩阵

print("📊 步骤 3: 因子统计量")
print(f"  因子均值 μ̂_f = {mu_f}")
print(f"  因子方差 Σ̂_f = {Sigma_f}")
print()

# ========== 步骤 4: 计算因子 Sharpe Ratio 平方 ==========
Sigma_f_inv = np.linalg.inv(Sigma_f)
SR2 = mu_f @ Sigma_f_inv @ mu_f  # μ'Σ⁻¹μ = SR² of tangency portfolio

print("📊 步骤 4: 因子 Sharpe Ratio 平方")
print(f"  公式: SR²_f = μ̂_f' Σ̂_f⁻¹ μ̂_f")
print(f"  SR²_f = {SR2:.6f}")
print(f"  SR_f (月度) = {np.sqrt(SR2):.4f}")
print(f"  SR_f (年化) = {np.sqrt(SR2) * np.sqrt(12):.4f}")
print()

# ========== 步骤 5: 计算 alpha 的加权平方和 ==========
Sigma_eps_inv = np.linalg.inv(Sigma_eps)
alpha_quad = alpha_hat @ Sigma_eps_inv @ alpha_hat  # α'Σε⁻¹α

print("📊 步骤 5: Alpha 加权平方和")
print(f"  公式: Q = α̂' Σ̂_ε⁻¹ α̂")
print(f"  Q = {alpha_quad:.6f}")
print(f"  💡 这个值越大，说明 alpha 整体越偏离零")
print()

# ========== 步骤 6: 组装 GRS 统计量 ==========
GRS = (T - N - K) / N * alpha_quad / (1 + SR2)
df1 = N
df2 = T - N - K
p_grs = 1 - stats.f.cdf(GRS, df1, df2)

print("📊 步骤 6: GRS 统计量")
print(f"  公式: GRS = (T-N-K)/N × Q / (1+SR²_f)")
print(f"  实际计算: GRS = ({T}-{N}-{K})/{N} × {alpha_quad:.4f} / (1+{SR2:.4f})")
print(f"              = {(T-N-K)/N:.4f} × {alpha_quad/(1+SR2):.4f}")
print(f"              = {GRS:.4f}")
print(f"")
print(f"  自由度: F({df1}, {df2})")
print(f"  P 值 = {p_grs:.6f}")
print()
if p_grs < 0.05:
    print(f"  ✅ GRS = {GRS:.3f}, P = {p_grs:.4f} < 0.05")
    print(f"  ✅ 拒绝 H₀：alpha 联合不为零 → 模型存在定价误差")
else:
    print(f"  ❌ GRS = {GRS:.3f}, P = {p_grs:.4f} ≥ 0.05")
    print(f"  ❌ 不能拒绝 H₀：无足够证据说明模型有定价误差")

In [ ]:
# ========== 封装 GRS 检验函数，后续复用 ==========
def grs_test(returns, factors):
    """
    GRS 检验 (Gibbons-Ross-Shanken 1989)
    
    参数:
        returns: (T, N) 检验资产超额收益矩阵
        factors: (T, K) 因子收益矩阵
    
    返回:
        dict: GRS统计量、P值、alpha向量、残差协方差等
    """
    T, N = returns.shape
    if factors.ndim == 1:
        factors = factors.reshape(-1, 1)
    K = factors.shape[1]
    
    # 时间序列回归
    X = sm.add_constant(factors)
    alpha_hat = np.zeros(N)
    residuals = np.zeros((T, N))
    
    for i in range(N):
        model = sm.OLS(returns[:, i], X).fit()
        alpha_hat[i] = model.params[0]
        residuals[:, i] = model.resid
    
    # 残差协方差矩阵 (ML 估计)
    Sigma_eps = (residuals.T @ residuals) / T
    
    # 因子统计量
    mu_f = factors.mean(axis=0)
    Sigma_f = np.cov(factors.T, ddof=0)
    if Sigma_f.ndim == 0:
        Sigma_f = np.array([[Sigma_f]])
    if Sigma_f.ndim == 1:
        Sigma_f = Sigma_f.reshape(1, 1)
    
    # GRS 统计量
    Sigma_f_inv = np.linalg.inv(Sigma_f)
    Sigma_eps_inv = np.linalg.inv(Sigma_eps)
    SR2 = mu_f @ Sigma_f_inv @ mu_f
    alpha_quad = alpha_hat @ Sigma_eps_inv @ alpha_hat
    
    GRS_stat = (T - N - K) / N * alpha_quad / (1 + SR2)
    df1, df2 = N, T - N - K
    p_value = 1 - stats.f.cdf(GRS_stat, df1, df2)
    
    return {
        'GRS': GRS_stat,
        'p_value': p_value,
        'df1': df1,
        'df2': df2,
        'alpha': alpha_hat,
        'Sigma_eps': Sigma_eps,
        'avg_abs_alpha': np.mean(np.abs(alpha_hat)),
        'SR2_factors': SR2
    }

# ========== 用封装函数验证手算结果 ==========
result_verify = grs_test(R, MKT)

print("🔬 封装函数验证：")
print(f"  手算 GRS = {GRS:.6f}")
print(f"  函数 GRS = {result_verify['GRS']:.6f}")
print(f"  手算 P值 = {p_grs:.6f}")
print(f"  函数 P值 = {result_verify['p_value']:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化：Alpha 分布 + GRS F 分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：Alpha 分布 ---
ax1 = axes[0]
colors_alpha = ['#2ecc71' if a > 0 else '#e74c3c' for a in alpha_hat]
bars = ax1.bar([f'Asset {i+1}' for i in range(N_micro)], alpha_hat,
               color=colors_alpha, edgecolor='black', alpha=0.85)
for bar, a in zip(bars, alpha_hat):
    va = 'bottom' if a >= 0 else 'top'
    offset = 0.02 if a >= 0 else -0.02
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{a:.3f}', ha='center', va=va, fontweight='bold', fontsize=10)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Test Asset', fontsize=12)
ax1.set_ylabel('Estimated Alpha (%)', fontsize=12)
ax1.set_title('Individual Alphas from CAPM Regression', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：GRS F 分布 ---
ax2 = axes[1]
x_f = np.linspace(0, max(GRS * 1.5, 5), 500)
f_dist = stats.f.pdf(x_f, df1, df2)
ax2.plot(x_f, f_dist, 'b-', linewidth=2, label=f'F({df1}, {df2}) distribution')
ax2.fill_between(x_f, f_dist, where=(x_f >= GRS), color='red', alpha=0.4,
                 label=f'P-value = {p_grs:.4f}')
ax2.axvline(x=GRS, color='red', linestyle='--', linewidth=2,
            label=f'GRS = {GRS:.3f}')

# F 临界值
f_crit = stats.f.ppf(0.95, df1, df2)
ax2.axvline(x=f_crit, color='orange', linestyle=':', linewidth=2,
            label=f'Critical value (5%) = {f_crit:.3f}')

ax2.set_xlabel('F Value', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('GRS Test: F Distribution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：5 个资产的估计 alpha。正值(绿色)和负值(红色)都偏离零")
print(f"  右图：GRS = {GRS:.3f} 在 F 分布中的位置。红色阴影 = P 值")
print(f"        GRS 越大 → P 值越小 → 越有证据拒绝 H₀ (alpha 联合为零)")

---

## 4. 模型比较实践：CAPM vs FF3 vs FF5

### 🎯 实验设计

**真实数据生成过程 (DGP)**：Fama-French 三因子模型

$$r_{i,t} = \alpha_i + \beta_{i,MKT} \cdot \text{MKT}_t + \beta_{i,SMB} \cdot \text{SMB}_t + \beta_{i,HML} \cdot \text{HML}_t + \varepsilon_{i,t}$$

其中 $\alpha_i = 0$（三因子模型是正确模型）。

**检验资产**：25 个投资组合（模拟 Fama-French 经典的 5×5 Size-BM 排序组合）

**预期结果**：

| 模型 | 因子 | 预期 GRS | 原因 |
|------|------|----------|------|
| CAPM | MKT | 大（拒绝） | 遗漏 SMB、HML → alpha 大 |
| FF3 | MKT, SMB, HML | 小（不拒绝） | 正确模型 → alpha ≈ 0 |
| FF5 | MKT, SMB, HML, RMW, CMA | 小（不拒绝） | 包含正确因子 + 多余因子 |

In [ ]:
# ========== 模拟 25 个 Size-BM 组合 + 多因子收益 ==========
np.random.seed(42)
T_full = 240    # 240 个月 (20 年) — 足够的样本量保证检验功效
N_full = 25     # 25 个检验资产 (5×5)

# --- 生成因子收益 ---
# 真实因子 (在 DGP 中)
# 设定较高的因子均值以确保 CAPM 遗漏效应明显
MKT_f = np.random.normal(0.6, 4.0, T_full)     # 市场因子
SMB_f = np.random.normal(0.4, 2.0, T_full)     # 市值因子: 均值 0.4%/月
HML_f = np.random.normal(0.5, 2.0, T_full)     # 价值因子: 均值 0.5%/月

# 额外因子 (不在 DGP 中，但与真实因子有微弱相关)
RMW_f = 0.1 * SMB_f + np.random.normal(0.15, 1.8, T_full)
CMA_f = 0.1 * HML_f + np.random.normal(0.10, 1.5, T_full)

# --- 设计 25 个组合的 beta 暴露 ---
# 模拟 5×5 Size-BM 排序：
#   行 = Size (小→大)，列 = BM (低→高)
#   小市值组 → 高 SMB beta
#   高 BM 组 → 高 HML beta

size_levels = np.array([1.2, 0.6, 0.0, -0.6, -1.2])    # SMB beta: 小→大
bm_levels = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])      # HML beta: 低→高

beta_MKT = np.ones(N_full)  # 所有组合的市场 beta ≈ 1
beta_SMB = np.zeros(N_full)
beta_HML = np.zeros(N_full)

portfolio_names = []
idx = 0
for i, s in enumerate(size_levels):
    for j, b in enumerate(bm_levels):
        beta_MKT[idx] = 0.8 + 0.4 * np.random.rand()  # MKT beta ∈ [0.8, 1.2]
        beta_SMB[idx] = s + 0.05 * np.random.randn()
        beta_HML[idx] = b + 0.05 * np.random.randn()
        portfolio_names.append(f'S{i+1}B{j+1}')
        idx += 1

# --- 生成收益 (真实模型 = FF3，alpha = 0) ---
R_full = np.zeros((T_full, N_full))
eps_std = 1.5  # 残差标准差 (较小，增强信号)

for i in range(N_full):
    R_full[:, i] = (beta_MKT[i] * MKT_f
                    + beta_SMB[i] * SMB_f
                    + beta_HML[i] * HML_f
                    + np.random.normal(0, eps_std, T_full))

print(f"📊 模拟参数：")
print(f"  检验资产数 N = {N_full} (5×5 Size-BM)")
print(f"  时间期数 T = {T_full}")
print(f"  真实 DGP: FF3 (MKT + SMB + HML)，所有 alpha = 0")
print(f"")
print(f"📊 因子统计量：")
print(f"  MKT: 均值={MKT_f.mean():.3f}%, 标准差={MKT_f.std():.3f}%")
print(f"  SMB: 均值={SMB_f.mean():.3f}%, 标准差={SMB_f.std():.3f}%")
print(f"  HML: 均值={HML_f.mean():.3f}%, 标准差={HML_f.std():.3f}%")
print(f"  RMW: 均值={RMW_f.mean():.3f}%, 标准差={RMW_f.std():.3f}% (不在 DGP 中)")
print(f"  CMA: 均值={CMA_f.mean():.3f}%, 标准差={CMA_f.std():.3f}% (不在 DGP 中)")
print(f"")
print(f"📊 前 5 个组合的 beta 设定：")
print(f"  {'组合':>6}  {'β_MKT':>8}  {'β_SMB':>8}  {'β_HML':>8}")
for i in range(5):
    print(f"  {portfolio_names[i]:>6}  {beta_MKT[i]:>8.3f}  {beta_SMB[i]:>8.3f}  {beta_HML[i]:>8.3f}")
print(f"  ... (共 {N_full} 个组合)")

In [ ]:
# ========== GRS 检验：CAPM ==========
factors_capm = MKT_f
result_capm = grs_test(R_full, factors_capm)

print("=" * 60)
print("📋 模型 1: CAPM (仅 MKT 因子)")
print("=" * 60)
print(f"  因子数 K = 1")
print(f"  GRS 统计量 = {result_capm['GRS']:.4f}")
print(f"  P 值 = {result_capm['p_value']:.6f}")
print(f"  F 分布: F({result_capm['df1']}, {result_capm['df2']})")
print(f"  平均 |α| = {result_capm['avg_abs_alpha']:.4f}%")
print(f"  因子 SR² = {result_capm['SR2_factors']:.4f}")
print()
if result_capm['p_value'] < 0.05:
    print(f"  ✅ 拒绝 H₀ → CAPM 存在显著定价误差")
    print(f"  💡 原因: CAPM 遗漏了 SMB 和 HML 因子")
else:
    print(f"  ❌ 不能拒绝 H₀")

# 展示部分 alpha
print(f"\n  前 10 个组合的 alpha:")
for i in range(10):
    print(f"    {portfolio_names[i]}: α = {result_capm['alpha'][i]:>7.3f}%")

In [ ]:
# ========== GRS 检验：Fama-French 三因子 ==========
factors_ff3 = np.column_stack([MKT_f, SMB_f, HML_f])
result_ff3 = grs_test(R_full, factors_ff3)

print("=" * 60)
print("📋 模型 2: Fama-French 三因子 (MKT + SMB + HML)")
print("=" * 60)
print(f"  因子数 K = 3")
print(f"  GRS 统计量 = {result_ff3['GRS']:.4f}")
print(f"  P 值 = {result_ff3['p_value']:.6f}")
print(f"  F 分布: F({result_ff3['df1']}, {result_ff3['df2']})")
print(f"  平均 |α| = {result_ff3['avg_abs_alpha']:.4f}%")
print(f"  因子 SR² = {result_ff3['SR2_factors']:.4f}")
print()
if result_ff3['p_value'] < 0.05:
    print(f"  ✅ 拒绝 H₀ → FF3 仍存在一些定价误差")
else:
    print(f"  ❌ 不能拒绝 H₀ → FF3 能较好解释收益截面差异")
    print(f"  💡 原因: FF3 是正确的 DGP，alpha 应接近零")

print(f"\n  前 10 个组合的 alpha:")
for i in range(10):
    print(f"    {portfolio_names[i]}: α = {result_ff3['alpha'][i]:>7.3f}%")

In [ ]:
# ========== GRS 检验：Fama-French 五因子 ==========
factors_ff5 = np.column_stack([MKT_f, SMB_f, HML_f, RMW_f, CMA_f])
result_ff5 = grs_test(R_full, factors_ff5)

print("=" * 60)
print("📋 模型 3: Fama-French 五因子 (MKT + SMB + HML + RMW + CMA)")
print("=" * 60)
print(f"  因子数 K = 5")
print(f"  GRS 统计量 = {result_ff5['GRS']:.4f}")
print(f"  P 值 = {result_ff5['p_value']:.6f}")
print(f"  F 分布: F({result_ff5['df1']}, {result_ff5['df2']})")
print(f"  平均 |α| = {result_ff5['avg_abs_alpha']:.4f}%")
print(f"  因子 SR² = {result_ff5['SR2_factors']:.4f}")
print()
if result_ff5['p_value'] < 0.05:
    print(f"  ✅ 拒绝 H₀ → FF5 仍存在一些定价误差")
else:
    print(f"  ❌ 不能拒绝 H₀ → FF5 能解释收益截面差异")
    print(f"  💡 FF5 包含了正确的 FF3 因子，额外的 RMW、CMA 不在 DGP 中")
    print(f"     多余因子不会改善解释力，但也不会显著恶化（自由度略微下降）")

print(f"\n  前 10 个组合的 alpha:")
for i in range(10):
    print(f"    {portfolio_names[i]}: α = {result_ff5['alpha'][i]:>7.3f}%")

In [ ]:
# ========== 三个模型的比较汇总 ==========
print("=" * 70)
print("📋 多因子模型比较汇总")
print("=" * 70)
print(f"  真实 DGP: Fama-French 三因子模型 (alpha = 0)")
print(f"  检验资产: {N_full} 个组合 (5×5 Size-BM)")
print(f"  样本期: {T_full} 个月")
print()

comparison = pd.DataFrame({
    'Model': ['CAPM', 'FF3', 'FF5'],
    'K (Factors)': [1, 3, 5],
    'GRS Statistic': [result_capm['GRS'], result_ff3['GRS'], result_ff5['GRS']],
    'P-value': [result_capm['p_value'], result_ff3['p_value'], result_ff5['p_value']],
    'Avg |alpha| (%)': [result_capm['avg_abs_alpha'], result_ff3['avg_abs_alpha'], result_ff5['avg_abs_alpha']],
    'Reject H0?': ['Yes' if r['p_value'] < 0.05 else 'No'
                   for r in [result_capm, result_ff3, result_ff5]]
})

print(comparison.to_string(index=False))
print()
print(f"🎯 结论：")
print(f"  1. CAPM: GRS 大、P 值小 → 拒绝 → 遗漏 SMB/HML 导致大量定价误差")
print(f"  2. FF3:  GRS 小、P 值大 → 不拒绝 → 正确模型，alpha 接近零")
print(f"  3. FF5:  GRS 小、P 值大 → 不拒绝 → 包含正确因子，多余因子影响不大")
print(f"")
print(f"  💡 FF3 和 FF5 表现相似，因为真实 DGP 是 FF3")
print(f"     在实际研究中，选择更简约的模型（FF3）更合理")
print("=" * 70)

In [ ]:
# ========== 可视化：模型比较 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['CAPM', 'FF3', 'FF5']
results = [result_capm, result_ff3, result_ff5]
model_colors = ['#e74c3c', '#2ecc71', '#3498db']

# --- 左图：GRS 统计量对比 ---
ax1 = axes[0]
grs_vals = [r['GRS'] for r in results]
bars = ax1.bar(models, grs_vals, color=model_colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, grs_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{v:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# F 临界值线
f_crit_full = stats.f.ppf(0.95, N_full, T_full - N_full - 5)
ax1.axhline(y=f_crit_full, color='orange', linestyle='--', linewidth=2,
            label=f'F critical (5%) \u2248 {f_crit_full:.2f}')
ax1.set_ylabel('GRS Statistic', fontsize=12)
ax1.set_title('GRS Test: Model Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 中图：平均 |alpha| 对比 ---
ax2 = axes[1]
abs_alpha_vals = [r['avg_abs_alpha'] for r in results]
bars2 = ax2.bar(models, abs_alpha_vals, color=model_colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars2, abs_alpha_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_ylabel('Average |Alpha| (%)', fontsize=12)
ax2.set_title('Average Absolute Alpha', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- 右图：各模型 alpha 的箱线图 ---
ax3 = axes[2]
alpha_data = [r['alpha'] for r in results]
bp = ax3.boxplot(alpha_data, tick_labels=models, patch_artist=True,
                  medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], model_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Alpha = 0')
ax3.set_ylabel('Alpha (%)', fontsize=12)
ax3.set_title('Alpha Distribution by Model', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：CAPM 的 GRS 远超临界值（模型被拒绝），FF3/FF5 的 GRS 较小")
print(f"  中图：CAPM 的平均 |alpha| 最大，说明定价误差最严重")
print(f"  右图：CAPM 的 alpha 分布宽且偏离零；FF3/FF5 的 alpha 紧密围绕零")

---

## 5. 平均绝对 Alpha ($|\alpha|$) 检验补充

### 📐 定义

$$\overline{|\alpha|} = \frac{1}{N} \sum_{i=1}^{N} |\hat{\alpha}_i|$$

### 💡 与 GRS 检验的对比

| 特性 | GRS 检验 | $\overline{|\alpha|}$ |
|------|----------|----------------------|
| 类型 | 正式统计检验 (有 P 值) | 描述性指标 (无 P 值) |
| 考虑残差相关性 | ✅ 通过 $\Sigma_\varepsilon^{-1}$ | ❌ 简单平均 |
| 考虑因子 Sharpe Ratio | ✅ 通过分母 | ❌ 不考虑 |
| 直觉性 | 较难理解 | 非常直观 |
| 适用场景 | 严格假设检验 | 初步模型筛选 |

### 📖 书中要点

> $\overline{|\alpha|}$ 虽然不是正式的统计检验，但因其直观性在实际研究中被广泛使用。
> 一个好的模型应该同时通过 GRS 检验（P 值大）且具有较小的 $\overline{|\alpha|}$。

In [ ]:
# ========== |alpha| 详细计算与比较 ==========
print("📊 步骤 1: 各模型 alpha 详细统计")
print("═" * 70)

for name, result in zip(models, results):
    alphas = result['alpha']
    print(f"\n  📋 {name}:")
    print(f"    平均 alpha:     {alphas.mean():>8.4f}%")
    print(f"    平均 |alpha|:   {np.mean(np.abs(alphas)):>8.4f}%")
    print(f"    alpha 标准差:   {alphas.std():>8.4f}%")
    print(f"    alpha 最小值:   {alphas.min():>8.4f}%")
    print(f"    alpha 最大值:   {alphas.max():>8.4f}%")
    print(f"    |alpha| > 0.5% 的资产数: {np.sum(np.abs(alphas) > 0.5)}/{N_full}")

print(f"\n" + "═" * 70)
print(f"\n📊 步骤 2: 模型比较汇总")
print(f"")

summary_df = pd.DataFrame({
    'Model': models,
    'GRS': [f"{r['GRS']:.3f}" for r in results],
    'P-value': [f"{r['p_value']:.4f}" for r in results],
    'Avg |alpha|': [f"{r['avg_abs_alpha']:.4f}%" for r in results],
    'Max |alpha|': [f"{np.max(np.abs(r['alpha'])):.4f}%" for r in results],
    'Verdict': [
        'Rejected' if r['p_value'] < 0.05 else 'Not Rejected'
        for r in results
    ]
})
print(summary_df.to_string(index=False))

print(f"\n💡 解读：")
print(f"  CAPM 的 |alpha| 最大 → 最多的定价误差 → 最差的模型")
print(f"  FF3 的 |alpha| 最小 → 最少的定价误差 → 最好且最简约的模型")
print(f"  FF5 的 |alpha| 与 FF3 相近 → 额外因子没有带来显著改善")

In [ ]:
# ========== 可视化：Alpha 热力图 (5×5 Size-BM) ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name, result, color in zip(axes, models, results, model_colors):
    # 将 25 个 alpha 重排为 5×5 矩阵
    alpha_matrix = result['alpha'].reshape(5, 5)
    
    # 确定统一的颜色范围
    vmax = max(np.abs(result_capm['alpha']).max(), 0.5)
    
    sns.heatmap(alpha_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-vmax, vmax=vmax,
                linewidths=1, ax=ax,
                xticklabels=[f'BM{j+1}' for j in range(5)],
                yticklabels=[f'S{i+1}' for i in range(5)],
                cbar_kws={'label': 'Alpha (%)'})
    
    ax.set_title(f'{name}: Alphas (Avg|\u03b1|={result["avg_abs_alpha"]:.3f}%)',
                fontsize=13, fontweight='bold')
    ax.set_xlabel('B/M Group', fontsize=11)
    ax.set_ylabel('Size Group', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  热力图中红色 = 正 alpha，蓝色 = 负 alpha，白色 = alpha ≈ 0")
print(f"  CAPM: 大量红蓝色块 → 严重的定价误差")
print(f"       小市值高BM组(左上)有正alpha，大市值低BM组(右下)有负alpha")
print(f"       这说明 CAPM 遗漏了 Size 和 Value 效应")
print(f"  FF3:  接近全白 → alpha 接近零 → 模型正确")
print(f"  FF5:  与 FF3 相似 → 额外因子没有显著改变 alpha")

In [ ]:
# ========== 综合比较可视化 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：GRS F 分布叠加三个模型的检验统计量 ---
ax1 = axes[0]
# 用 FF5 的自由度画 F 分布 (作为参考)
df1_ref, df2_ref = N_full, T_full - N_full - 5
x_f = np.linspace(0, max(result_capm['GRS'] * 1.2, 6), 500)
f_pdf = stats.f.pdf(x_f, df1_ref, df2_ref)
ax1.plot(x_f, f_pdf, 'b-', linewidth=2, alpha=0.5, label=f'F({df1_ref},{df2_ref})')
ax1.fill_between(x_f, f_pdf, alpha=0.1, color='blue')

for name, result, color in zip(models, results, model_colors):
    ax1.axvline(x=result['GRS'], color=color, linestyle='--', linewidth=2.5,
                label=f"{name}: GRS={result['GRS']:.2f} (P={result['p_value']:.3f})")

f_crit_ref = stats.f.ppf(0.95, df1_ref, df2_ref)
ax1.axvline(x=f_crit_ref, color='black', linestyle=':', linewidth=1.5,
            label=f'Critical (5%) = {f_crit_ref:.2f}')

ax1.set_xlabel('F Value', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('GRS Statistics on F Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(alpha=0.3)

# --- 右图：GRS vs |alpha| 散点图 ---
ax2 = axes[1]
for name, result, color in zip(models, results, model_colors):
    ax2.scatter(result['avg_abs_alpha'], result['GRS'],
                s=200, c=color, edgecolors='black', zorder=5, label=name)
    ax2.annotate(f"{name}\nGRS={result['GRS']:.2f}\n|\u03b1|={result['avg_abs_alpha']:.3f}%",
                 (result['avg_abs_alpha'], result['GRS']),
                 textcoords='offset points', xytext=(15, 5), fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

ax2.set_xlabel('Average |Alpha| (%)', fontsize=12)
ax2.set_ylabel('GRS Statistic', fontsize=12)
ax2.set_title('Model Comparison: GRS vs |Alpha|', fontsize=14, fontweight='bold')

# 标注"好模型"区域
ax2.annotate('Better Models\n(lower GRS, lower |\u03b1|)', xy=(0.02, 0.02),
             xycoords='axes fraction', fontsize=10, color='green',
             fontweight='bold', fontstyle='italic')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：三个模型的 GRS 统计量在 F 分布上的位置")
print(f"        CAPM 远超临界值，FF3/FF5 在临界值左侧")
print(f"  右图：散点图的左下角 = 好模型 (低 GRS + 低 |alpha|)")
print(f"        FF3 在最左下方 → 最好的模型 (正确且简约)")

---

## 6. 核心概念回顾

### 📌 GRS 检验 (Gibbons-Ross-Shanken Test)
- **定义**: 联合检验所有检验资产相对于某因子模型的 alpha 是否同时为零
- **公式**: $\text{GRS} = \frac{T-N-K}{N} \cdot \frac{\hat{\alpha}' \hat{\Sigma}_\varepsilon^{-1} \hat{\alpha}}{1 + \hat{\mu}_f' \hat{\Sigma}_f^{-1} \hat{\mu}_f} \sim F(N, T-N-K)$
- **含义**: GRS 越大 → alpha 整体偏离零越严重 → 模型越差
- **判断**: P 值 < 0.05 → 拒绝 H₀ → 模型存在显著定价误差

### 📌 平均绝对 Alpha ($\overline{|\alpha|}$)
- **定义**: 所有检验资产 alpha 绝对值的简单平均
- **公式**: $\overline{|\alpha|} = \frac{1}{N} \sum_{i=1}^{N} |\hat{\alpha}_i|$
- **含义**: 衡量模型定价误差的平均大小
- **判断**: 值越小 → 模型解释力越好

### 📌 多重检验问题 (Multiple Testing Problem)
- **定义**: 同时进行多个假设检验时，至少一个误判的概率急剧上升
- **公式**: $P(\text{至少一个误判}) = 1 - (1-\alpha)^N$
- **含义**: 逐个检验 alpha 不可靠；应使用 GRS 联合检验

### 📌 因子 Sharpe Ratio
- **定义**: 因子组合切点组合的最大 Sharpe Ratio
- **公式**: $\text{SR}^2_f = \hat{\mu}_f' \hat{\Sigma}_f^{-1} \hat{\mu}_f$
- **含义**: GRS 分母中的 $(1 + \text{SR}^2_f)$ 调整了因子本身的定价能力

### 🔗 完整流程
```
选择检验资产 → 对每个模型做 N 个时间序列回归 → 收集 alpha 和残差
    ↓                                                      ↓
  5×5 组合                                        计算 Σ_ε 和因子统计量
                                                          ↓
                                          组装 GRS 统计量 + 计算 P 值
                                                          ↓
                                          比较各模型 GRS 和 |alpha|
                                                          ↓
                                        GRS 小 + |alpha| 小 = 好模型
```

### 📝 检验指标汇总

| 指标 | 含义 | 好模型应该 |
|------|------|------------|
| GRS 统计量 | alpha 联合偏离零的程度 | 小 (P 值大) |
| P 值 | 拒绝 H₀ 的证据强度 | > 0.05 (不拒绝) |
| $\overline{\|\alpha\|}$ | 平均定价误差大小 | 小 |
| $\max\|\alpha_i\|$ | 最大单个定价误差 | 小 |

---

## 7. 常见误区

### ❌ 误区 1: 逐个检验 alpha 就能判断模型好坏
**✓ 正确理解**: 逐个检验存在多重检验问题——25 个资产中有 72% 的概率至少出现一个"假阳性"。必须使用 GRS 联合检验，它一次性考虑所有 alpha 并给出一个统一的 P 值。

### ❌ 误区 2: GRS 不被拒绝就说明模型是"正确的"
**✓ 正确理解**: 不拒绝 H₀ 只是说"没有足够证据拒绝"，不等于"模型为真"。可能是样本量不够大，检验功效不足。此外，GRS 的假设前提（正态分布残差、恒定 beta）在实际中可能不完全满足。

### ❌ 误区 3: 因子越多模型越好
**✓ 正确理解**: 增加因子会消耗自由度（GRS 分母中 $T-N-K$ 减小），如果新因子对解释收益没有帮助，GRS 反而可能变大。模型比较应遵循简约原则——在解释力相近时选择因子更少的模型。

### ❌ 误区 4: 只看 GRS 不看 $|\alpha|$，或只看 $|\alpha|$ 不看 GRS
**✓ 正确理解**: 两个指标互补。GRS 是正式统计检验（考虑了残差相关性和因子 Sharpe Ratio），$|\alpha|$ 直观反映定价误差大小。理想情况下两者应同时使用：GRS 不被拒绝 + $|\alpha|$ 小 = 好模型。

### ❌ 误区 5: GRS 检验可以比较任意两个"不同检验资产"的模型
**✓ 正确理解**: GRS 检验要求不同模型使用**相同的检验资产集**。你不能拿 CAPM 在 25 个 Size-BM 组合上的 GRS 去和 FF3 在 10 个行业组合上的 GRS 直接比较。检验资产必须保持一致，才能公平比较模型。